## RAG (Retrieval-Augmented Generation) Implementation

#### Configure Gemini API

In [45]:
from dotenv import load_dotenv
from google import genai
import os

load_dotenv()
API_KEY = os.getenv("GEMINI_API_KEY")


client = genai.Client(api_key=API_KEY)

#### Read Pdf File

In [46]:
from pypdf import PdfReader

def load_pdf(file_path):
  reader = PdfReader(file_path)
  text = ""
  
  for page in reader.pages:
    text += page.extract_text()
    
  return text


# File uploading
file_text = load_pdf("./data/UjjwalKumar-RESUME.pdf")

print(file_text[:200])

Ujjwal kumar
/envel⌢peujjwal.kumar.id@gmail.com♂phone-alt7061845104/linkedinLinkedin/githubGithubὑ7Portfolio
Education
B.Tech in Electronics & Communication 2022 – 2026
Maharaja Agrasen Institute of T


#### Split Text into Chunks

In [47]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
  chunk_size=300,
  chunk_overlap=100
)

chunks = splitter.split_text(file_text)

print("Total chunks :", len(chunks))
print(chunks[0])

Total chunks : 10
Ujjwal kumar
/envel⌢peujjwal.kumar.id@gmail.com♂phone-alt7061845104/linkedinLinkedin/githubGithubὑ7Portfolio
Education
B.Tech in Electronics & Communication 2022 – 2026
Maharaja Agrasen Institute of Technology , Delhi
• CGPA: 7.8/10.0 [Marksheet]
Senior Secondary (XII), CBSE 2021


#### Embeddings

In [49]:
from google.genai import types
from langchain_core.embeddings import Embeddings

def get_embedding(text):
  result = client.models.embed_content(
    model="gemini-embedding-001",
    contents=file_text,
    config=types.EmbedContentConfig(output_dimensionality=384)
  )
  
  return result.embeddings[0].values

class GeminiEmbeddings(Embeddings):
  def embed_documents(self, texts):
    return [get_embedding(t) for t in texts]
  
  def embed_query(self, text):
    return get_embedding(text)

#### Configure VectorDB (ChromaDB)

In [50]:
from langchain_community.vectorstores import Chroma

vectorstore = Chroma.from_texts(
  texts=chunks,
  embedding=GeminiEmbeddings()
)

#### Retrieve from VectorDB

In [51]:
def retrieve(query):
  docs = vectorstore.similarity_search(query, k=3)
  return "\n\n".join([d.page_content for d in docs])

#### Generate Response from LLM

In [56]:
def ask_gemini(context, question):
  prompt = f"""
  You are answering questions based ONLY on the resume.
  
  Context: {context}
  Question: {question}
  
  If answer is not in resume, say "Not mentioned in resume." 
  """
  
  response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=prompt
  )
  
  return response.text

#### Sample Test

In [57]:
question = "Summarize the whole resume in 200 words."

context = retrieve(question)

answer = ask_gemini(context, question)

print("Response: \n", answer)

Response: 
 Ujjwal Kumar is a final-year B.Tech student in Electronics & Communication at Maharaja Agrasen Institute of Technology, expected to graduate in 2026, holding a CGPA of 7.8/10.0. Ujjwal also completed Senior Secondary (XII) CBSE in 2021.

Ujjwal possesses hands-on experience in full-stack development, is comfortable with data structures and algorithms in C++, and is passionate about solving real-world problems with clean, efficient code.

Key projects include a "Community App," where Ujjwal integrated Google Authentication for secure user onboarding and enabled users to create and join communities with member and admin role controls. This project utilized React.js, Express.js, PostgreSQL, and TypeScript. Another project, "BrainBox," also has a GitHub link.

For professional development, Ujjwal completed a "Web Development Course" from Udemy. Contact information, including email (ujjwal.kumar.id@gmail.com), phone (7061845104), LinkedIn, GitHub, and a portfolio link are readil